## 1. Import Necessary Liberaries

In [113]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler

## 2. Data Collection

In [114]:
weather_data = pd.read_csv("environmental_data.csv")
weather_data

,Unnamed: 0,Ozone,Solar.R,Wind,Temp C,Month,Day,Year,Temp,Weather
0,1,41.0,190.0,7.4,67,5,1,2010,67,S
1,2,36.0,118.0,8.0,72,5,2,2010,72,C
2,3,12.0,149.0,12.6,74,5,3,2010,74,PS
3,4,18.0,313.0,11.5,62,5,4,2010,62,S
4,5,NaN,NaN,14.3,56,5,5,2010,56,S
...,...,...,...,...,...,...,...,...,...,...
153,154,41.0,190.0,7.4,67,5,1,2010,67,C
154,155,30.0,193.0,6.9,70,9,26,2010,70,PS
155,156,NaN,145.0,13.2,77,9,27,2010,77,S
156,157,14.0,191.0,14.3,75,9,28,2010,75,S


## 3. Data Understand

### 3.1. Perform Intial Analysis on the Data

In [115]:
weather_data.shape

(158, 10)

In [116]:
weather_data.isna().sum()

Unnamed: 0     0
Ozone         38
Solar.R        7
Wind           0
Temp C         0
Month          0
Day            0
Year           0
Temp           0
Weather        3
dtype: int64

In [117]:
weather_data.dtypes

Unnamed: 0      int64
Ozone         float64
Solar.R       float64
Wind          float64
Temp C         object
Month          object
Day             int64
Year            int64
Temp            int64
Weather        object
dtype: object

In [118]:
weather_data.describe(include="all")

,Unnamed: 0,Ozone,Solar.R,Wind,Temp C,Month,Day,Year,Temp,Weather
count,158.000000,120.000000,151.000000,158.000000,158,158,158.000000,158.0,158.000000,155
unique,NaN,NaN,NaN,NaN,41,6,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,81,9,NaN,NaN,NaN,S
freq,NaN,NaN,NaN,NaN,11,34,NaN,NaN,NaN,59
mean,79.500000,41.583333,185.403974,9.957595,NaN,NaN,16.006329,2010.0,77.727848,NaN
std,45.754781,32.620709,88.723103,3.511261,NaN,NaN,8.997166,0.0,9.377877,NaN
min,1.000000,1.000000,7.000000,1.700000,NaN,NaN,1.000000,2010.0,56.000000,NaN
25%,40.250000,18.000000,119.000000,7.400000,NaN,NaN,8.000000,2010.0,72.000000,NaN
50%,79.500000,30.500000,197.000000,9.700000,NaN,NaN,16.000000,2010.0,78.500000,NaN
75%,118.750000,61.500000,257.000000,11.875000,NaN,NaN,24.000000,2010.0,84.000000,NaN


In [119]:
weather_data["Weather"].unique()

array(['S', 'C', 'PS', nan], dtype=object)

In [120]:
weather_data["Weather"].value_counts(dropna = False)

Weather
S      59
C      49
PS     47
NaN     3
Name: count, dtype: int64

## 4. Data Preparation

### 4.1. Data Cleaning 

#### 4.1.1 Remove Unnecessary Columns

In [121]:
weather_data.drop(labels = ["Unnamed: 0","Temp C"],axis= 1, inplace = True)
weather_data

,Ozone,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,41.0,190.0,7.4,5,1,2010,67,S
1,36.0,118.0,8.0,5,2,2010,72,C
2,12.0,149.0,12.6,5,3,2010,74,PS
3,18.0,313.0,11.5,5,4,2010,62,S
4,NaN,NaN,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...,...
153,41.0,190.0,7.4,5,1,2010,67,C
154,30.0,193.0,6.9,9,26,2010,70,PS
155,NaN,145.0,13.2,9,27,2010,77,S
156,14.0,191.0,14.3,9,28,2010,75,S


In [122]:
weather_data.isna().sum()

Ozone      38
Solar.R     7
Wind        0
Month       0
Day         0
Year        0
Temp        0
Weather     3
dtype: int64

#### 4.1.2.Remove most Null value column - Ozone

In [123]:
38/158*100

24.050632911392405

In [124]:
del weather_data["Ozone"]

In [125]:
weather_data

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,NaN,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.0,7.4,5,1,2010,67,C
154,193.0,6.9,9,26,2010,70,PS
155,145.0,13.2,9,27,2010,77,S
156,191.0,14.3,9,28,2010,75,S


#### 4.1.3. Update Null value using fillna with mean, median, mode Imputation

In [126]:
weather_data.isna().sum()

Solar.R    7
Wind       0
Month      0
Day        0
Year       0
Temp       0
Weather    3
dtype: int64

In [127]:
weather_data[weather_data["Solar.R"].isna()]

,Solar.R,Wind,Month,Day,Year,Temp,Weather
4,NaN,14.3,5,5,2010,56,S
5,NaN,14.9,5,6,2010,66,C
10,NaN,6.9,5,11,2010,74,C
26,NaN,8.0,5,27,2010,57,PS
95,NaN,6.9,8,4,2010,86,NaN
96,NaN,7.4,8,5,2010,85,S
97,NaN,4.6,8,6,2010,87,C


In [128]:
weather_data.groupby("Weather")["Solar.R"].mean().round()

Weather
C     192.0
PS    174.0
S     193.0
Name: Solar.R, dtype: float64

In [129]:
weather_data["Solar.R"] = weather_data.groupby("Weather")["Solar.R"].transform(lambda x:x.fillna(x.mean().round()))
weather_data

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,193.0,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.0,7.4,5,1,2010,67,C
154,193.0,6.9,9,26,2010,70,PS
155,145.0,13.2,9,27,2010,77,S
156,191.0,14.3,9,28,2010,75,S


In [130]:
weather_data.isna().sum()

Solar.R    3
Wind       0
Month      0
Day        0
Year       0
Temp       0
Weather    3
dtype: int64

#### 4.1.4. Drop the Null values

In [131]:
weather_data[weather_data["Solar.R"].isna()]

,Solar.R,Wind,Month,Day,Year,Temp,Weather
87,NaN,12.0,7,27,2010,86,NaN
93,NaN,13.8,8,2,2010,81,NaN
95,NaN,6.9,8,4,2010,86,NaN


In [132]:
weather_data.dropna(inplace = True)

In [133]:
weather_data.isna().sum()

Solar.R    0
Wind       0
Month      0
Day        0
Year       0
Temp       0
Weather    0
dtype: int64

#### 4.1.4 . Change the datatypes

In [134]:
weather_data.dtypes

Solar.R    float64
Wind       float64
Month       object
Day          int64
Year         int64
Temp         int64
Weather     object
dtype: object

In [135]:
weather_data["Month"].unique()

array(['5', 'May', '6', '7', '8', '9'], dtype=object)

In [136]:
weather_data.head(25)

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,193.0,14.3,5,5,2010,56,S
5,192.0,14.9,5,6,2010,66,C
6,299.0,8.6,5,7,2010,65,PS
7,99.0,13.8,5,8,2010,59,C
8,19.0,20.1,5,9,2010,61,PS
9,194.0,8.6,5,10,2010,69,S


In [137]:
weather_data["Month"].replace(to_replace="May",value =5,inplace = True)

C:\Users\HP\AppData\Local\Temp\ipykernel_3804\4249371337.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  weather_data["Month"].replace(to_replace="May",value =5,inplace = True)


In [138]:
weather_data.head(30)

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,193.0,14.3,5,5,2010,56,S
5,192.0,14.9,5,6,2010,66,C
6,299.0,8.6,5,7,2010,65,PS
7,99.0,13.8,5,8,2010,59,C
8,19.0,20.1,5,9,2010,61,PS
9,194.0,8.6,5,10,2010,69,S


In [140]:
weather_data["Month"] = weather_data["Month"].astype("int64")

In [141]:
weather_data.dtypes

Solar.R    float64
Wind       float64
Month        int64
Day          int64
Year         int64
Temp         int64
Weather     object
dtype: object

### 4.2. Data Transform

In [151]:
weather_data_copy_02 = weather_data.copy()

In [152]:
weather_data_copy_02

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,193.0,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.0,7.4,5,1,2010,67,C
154,193.0,6.9,9,26,2010,70,PS
155,145.0,13.2,9,27,2010,77,S
156,191.0,14.3,9,28,2010,75,S


In [ ]:
weather_data_copy_01["Weather"]

In [154]:
le_encoder = LabelEncoder()

weather_data_copy_02["Weather"] = le_encoder.fit_transform(weather_data_copy_02["Weather"])

In [155]:
weather_data_copy_02

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,2
1,118.0,8.0,5,2,2010,72,0
2,149.0,12.6,5,3,2010,74,1
3,313.0,11.5,5,4,2010,62,2
4,193.0,14.3,5,5,2010,56,2
...,...,...,...,...,...,...,...
153,190.0,7.4,5,1,2010,67,0
154,193.0,6.9,9,26,2010,70,1
155,145.0,13.2,9,27,2010,77,2
156,191.0,14.3,9,28,2010,75,2
